# 00b. Macro & Market Regime 데이터 구축

## 📋 개요
종목별 데이터 수집에 앞서, 한국 시장의 전역적(Global) 환경 변수인 거시 경제 지표와 Market Regime(강세/약세/보합장) 시그널을 계산하여 별도로 저장합니다.

## ✨ 핵심 파이프라인 및 업데이트
- **3단계 Fallback 수집 (v3.8.1)**: `FinanceDataReader` API 호출 실패 시 `data/99_meta/*.csv`의 로컬 백업 데이터를 자동으로 파싱하여 파이프라인 중단을 방지합니다.
- **KOSPI 거래일 동기화**: 모든 해외 지표(S&P500, VIX, 환율 등)는 KOSPI 개장일을 기준으로 재인덱싱되며, KOSPI 영업일인데 해외 지표가 없는 날(해외 휴장)은 직전 거래일 값으로 `ffill` 처리됩니다.
- **Regime 판단**: KOSPI 200일 이동평균선과 장단기 변동성을 종합하여 Bull(1), Bear(-1), Neutral(0) 상태를 산출합니다.

## ⚠️ end 당일 ffill 현상 안내
`fetch_end = base_end`로 설정하면 API 특성상 end 당일 데이터가 직전 거래일 값으로 ffill 처리되는 경우가 있습니다.
이 현상이 발생하면 저장 셀 하단의 **수동 보정 블록** 주석을 해제하고 실제값을 직접 입력하여 보정합니다.

In [ ]:
import re
import pandas as pd
import numpy as np
import FinanceDataReader as fdr
from datetime import datetime, timedelta
from pathlib import Path

from src.utils.config import load_config, ProjectPaths
cfg = load_config()
paths = ProjectPaths.from_config(cfg)
macro_filepath = paths.get_macro_parquet()

print(f"📁 매크로 데이터 저장 경로: {macro_filepath}")

## 1️⃣ Fallback 헬퍼 정의
API 장애 시 `data/99_meta/*.csv`에서 데이터를 읽어오는 헬퍼 함수입니다.

CSV 날짜 컬럼이 `'2022. 7. 7 오후 3:30:00'` 형태인 경우에도 정규식으로 날짜 부분만 추출합니다.
동일 날짜에 여러 행이 존재하면 마지막 행(장 마감 기준)을 사용합니다.

In [ ]:
def _parse_csv_date(date_str: str) -> pd.Timestamp | None:
    """
    '2022. 7. 7 오후 3:30:00' 형태의 날짜 문자열에서 날짜(Date)만 추출.
    숫자로 시작하는 'YYYY. M. D' 부분만 파싱하고 시각은 무시.
    """
    m = re.match(r'(\d{4})\s*\.\s*(\d{1,2})\s*\.\s*(\d{1,2})', str(date_str))
    if m:
        return pd.Timestamp(f"{m.group(1)}-{m.group(2).zfill(2)}-{m.group(3).zfill(2)}")
    return None


def load_fallback_csv(indicator: str, meta_dir: Path) -> pd.Series:
    """
    data/99_meta/{indicator}.csv 를 읽어 날짜 인덱스의 Close Series 반환.

    처리 규칙:
    - Date 컬럼에서 날짜만 추출 (시각 제거)
    - 같은 날짜가 여러 행일 경우 마지막 행(장 마감 기준) 사용
    """
    csv_path = meta_dir / f"{indicator}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Fallback CSV 없음: {csv_path}")

    df = pd.read_csv(csv_path)

    # 날짜 파싱: 'YYYY. M. D 오전/오후 H:MM:SS' → date only
    df['Date'] = df['Date'].apply(_parse_csv_date)
    df = df.dropna(subset=['Date'])

    # 같은 날짜가 여러 행인 경우 마지막 값(장 마감) 사용
    df = df.sort_values('Date').groupby('Date', sort=True).last()

    series = df['Close'].astype(float)
    series.index = pd.to_datetime(series.index)
    series.index.name = 'Date'
    series.name = indicator
    return series


print("✅ Fallback 헬퍼 함수 정의 완료")

## 2️⃣ 매크로 지표 수집 및 KOSPI 거래일 정렬
KOSPI를 기준 인덱스로 먼저 확보한 뒤, 해외 지표(S&P500, USD/KRW, VIX)를 수집합니다.
해외 지표가 없는 KOSPI 영업일(해외 휴장)은 `ffill`로 직전 거래일 값을 채웁니다.

In [ ]:
base_start = pd.to_datetime(cfg['data_collection']['start_date'])
base_end   = pd.to_datetime(cfg['data_collection']['end_date'])
fetch_start = (base_start - timedelta(days=364)).strftime('%Y-%m-%d')
fetch_end   = base_end.strftime('%Y-%m-%d')

print(f"📥 데이터 수집 중... ({fetch_start} ~ {fetch_end})")

macro_raw = {}
sources   = {}

# ── KOSPI (기준 인덱스) ────────────────────────────────────────────────────
# KOSPI는 이후 모든 지표의 날짜 기준을 결정하므로 별도로 먼저 수집한다.
try:
    macro_raw['kospi'] = fdr.DataReader('KS11', fetch_start, fetch_end)['Close']
    if macro_raw['kospi'].empty:
        raise ValueError("빈 데이터")
    sources['kospi'] = 'API'
except Exception as e:
    print(f"  ⚠️  KOSPI API 실패 ({e}) → CSV fallback")
    macro_raw['kospi'] = load_fallback_csv('kospi', meta_dir)
    sources['kospi'] = 'CSV'

kospi_series = macro_raw['kospi'].copy()
kospi_series.index = pd.to_datetime(kospi_series.index)
kospi_series = kospi_series[
    (kospi_series.index >= fetch_start) & (kospi_series.index <= fetch_end)
]
kospi_index = kospi_series.index

# ── 해외 지표 수집 (symbols 딕셔너리 루프) ────────────────────────────────
symbols = {'sp500': ('US500', 'Close'), 'usd_krw': ('USD/KRW', 'Close'), 'vix': ('FRED:VIXCLS', 'VIXCLS')}

for key, (symbol, col_name) in symbols.items():
    try:
        series = fdr.DataReader(symbol, fetch_start, fetch_end)[col_name]
        if series.empty:
            raise ValueError("빈 데이터")
        macro_raw[key] = series
        sources[key] = 'API'
    except Exception as e:
        print(f"  ⚠️  {key.upper()} API 실패 ({e}) → CSV fallback")
        try:
            macro_raw[key] = load_fallback_csv(key, meta_dir)
            sources[key] = 'CSV'
        except FileNotFoundError:
            print(f"  ⚠️  {key.upper()} CSV도 없음 → 빈 Series 사용")
            macro_raw[key] = pd.Series(dtype=float)
            sources[key] = 'EMPTY'

# ── KOSPI 거래일 기준 인덱스 정렬 ────────────────────────────────────────
# KOSPI 영업일에 해외 지표가 없으면(해외 휴장) 직전 거래일 값으로 ffill 처리한다.
aligned = {'kospi': kospi_series}

for key in ['sp500', 'usd_krw', 'vix']:
    s = macro_raw[key].copy()
    s.index = pd.to_datetime(s.index)
    if s.empty:
        aligned[key] = pd.Series(np.nan, index=kospi_index, name=key)
        continue
    combined_index = s.index.union(kospi_index).sort_values()
    aligned[key] = s.reindex(combined_index).ffill().reindex(kospi_index)

df_macro = pd.DataFrame(aligned)
df_macro.index.name = 'Date'

# ── [수동 보정] API ffill 간극 수정 ───────────────────────────────────────
# fetch_end = base_end 설정 시, API 특성에 따라 end 당일 데이터가 ffill 처리될 수 있다.
# tail() 출력에서 마지막 행 값이 전일과 동일하다면 ffill 현상이 발생한 것이다.
# 해당 현상 확인 시 아래 주석을 해제하고 실제값을 입력한다.
# kospi는 이후 Regime 계산(MA200, 변동성)에 사용되므로 반드시 이 시점에 보정해야 한다.
# 재실행 전 반드시 다시 주석 처리할 것 (이전 실제값으로 정상값을 덮어쓰는 것을 방지).
#
# df_macro.loc[df_macro.index[-1], 'sp500']   = 6528.52
# df_macro.loc[df_macro.index[-1], 'usd_krw'] = 1516.13
# df_macro.loc[df_macro.index[-1], 'vix']     = 30.61
# ─────────────────────────────────────────────────────────────────────────

print(f"\n✅ 수집 및 KOSPI 정렬 완료: {len(df_macro):,} 거래일")
print("   수집 경로:")
for k, v in sources.items():
    print(f"     {k:<10}: {v}")
print(df_macro.tail())

## 3️⃣ Market Regime 및 파생 피처 계산

In [ ]:
print("⚙️ Market Regime 및 파생 피처 계산 중...")

# 1. KOSPI 기술적 지표 계산
df_macro['kospi_ma200'] = df_macro['kospi'].rolling(window=200).mean()
df_macro['kospi_vol20'] = df_macro['kospi'].pct_change().rolling(window=20).std()

# 동적 임계값: 최근 1년(250거래일) 변동성 중위수
vol_median = df_macro['kospi_vol20'].rolling(window=250).median()

# 2. Regime 판단 로직
#   - Bull (1) : 주가가 200일선 위
#   - Bear (-1): 주가가 200일선 아래 & 단기 변동성이 장기 중위수보다 큼 (투매 장세)
#   - Neutral (0): 그 외 횡보장
conditions = [
    (df_macro['kospi'] > df_macro['kospi_ma200']),
    (df_macro['kospi'] < df_macro['kospi_ma200']) & (df_macro['kospi_vol20'] > vol_median)
]
choices = [1, -1]
df_macro['market_regime'] = np.select(conditions, choices, default=0)

# 3. 미국 시장 수익률 (한국 시초가에 영향을 주는 전일 미국 시장 수익률)
df_macro['us_return_1d'] = df_macro['sp500'].pct_change().shift(1)

# 4. 결측치 제거 (초기 250일분 이동평균 계산 구간)
df_macro = df_macro.dropna()

print("✅ 계산 완료. Regime 분포:")
print(df_macro['market_regime'].value_counts(normalize=True).map('{:.1%}'.format))

## 4️⃣ 데이터 저장

In [ ]:
df_macro_final = df_macro.reset_index()
if 'Date' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'Date': 'date'})
elif 'index' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'index': 'date'})

final_cols = ['date', 'kospi', 'usd_krw', 'vix', 'us_return_1d', 'market_regime']
df_save = df_macro_final[final_cols].copy()


paths.ensure_dirs()
df_save.to_parquet(macro_filepath, index=False)
csv_filepath = paths.get_macro_csv()
df_save.to_csv(csv_filepath, index=False)

print(f"💾 매크로 데이터 저장 완료 → {macro_filepath}")
print(f"   총 데이터: {len(df_save):,}일")
display(df_save.tail())